# 2 Local Debugging Tutor
######
In this notebook, we will build a simple debugging tutor using a small local language model.

## Learning goals
- Load a GGUF model with `llama-cpp-python`
- Define a short system prompt for a Socratic debugging tutor
- Launch a simple Gradio chat interface

This tutor should not give the final answer. Instead, it should guide the student with short questions and hints.


## 1. Import the packages

We need three things for this notebook:

- `os` to work with file paths
- `Llama` from `llama-cpp-python` to load the local GGUF model
- `gradio` to build a simple chat interface


In [1]:
import os

try:
    from llama_cpp import Llama
except ImportError:
    %pip install llama-cpp-python
    from llama_cpp import Llama

try:
    import gradio as gr
except ImportError:
    %pip install gradio
    import gradio as gr


## 2. Load the local model

This cell loads a GGUF model from the shared folder.

The settings below are intentionally simple:
- `n_gpu_layers=0` keeps the notebook CPU-only
- `n_ctx=1024` gives a small context window
- `n_threads=4` is a safe starting point on a shared JupyterHub


In [2]:
# Change this to match where your local GGUF model is stored.
model_file_path = "/home/jovyan/shared/DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"

# Keep the context window in its own variable so we can reuse it later in the UI.
context_window = 1024

local_model = Llama(
    model_path=model_file_path,
    n_gpu_layers=0,   # CPU only
    n_batch=64,
    n_ctx=context_window,
    n_threads=4,
    verbose=False,
)

local_model_name = os.path.basename(model_file_path).replace(".gguf", "")
print("Model loaded:", local_model_name)


llama_context: n_ctx_per_seq (1024) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Model loaded: DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M


## 3. Write the tutor prompt

We want the model to act like a Socratic debugging tutor.
It should not give answers or corrected code.
Instead, it should guide the student with one small question or hint at a time.


In [3]:
system_prompt = """
You are a Socratic debugging tutor.

Rules:
- Do not give answers or corrected code.
- Ask one guiding question at a time.
- Give only one small hint or check.
- Keep it short and clear.
- Focus on one issue at a time.
- End with one short question.
""".strip()


## 4. Define the chat function

This function sends three things to the model:
1. the system prompt
2. the previous chat history
3. the new student message

The model then returns a short tutoring response.


In [4]:
def chat_with_local_model(message, history):
    messages = [{"role": "system", "content": system_prompt}]

    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})

    response = local_model.create_chat_completion(
        messages=messages,
        max_tokens=120,
        temperature=0.4,
    )

    return response["choices"][0]["message"]["content"].strip()


## 5. Launch the Gradio app

Now we wrap the tutor in a Gradio chat interface.
Students can ask a debugging question and get short guided help.


In [6]:
demo = gr.ChatInterface(
    fn=chat_with_local_model,
    title="Local Socratic Debugging Tutor",
    description=f"Model: {local_model_name} | Context window: {context_window}",
    examples=[
        "My Python loop only prints the first item. What should I check?",
        "I get an IndexError in my list code. Can you help me debug it?",
        "My function returns None when I expect a number.",
    ],
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://67bb2a1596a91c735d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Reflection questions

- How does the tutor respond differently from a normal code assistant?
- What makes a debugging hint helpful without giving away the answer?
- What kinds of bugs are still hard for a small local model?
